# What files are available in the wind bucket?

In [ ]:
import s3fs
fs = s3fs.S3FileSystem(anon=True, client_kwargs={"endpoint_url": "https://mghp.osn.xsede.org"})
fs.ls("cnh-bucket-1/llc_wind/kerchunk_files/")[:]

# What is the date at a given iteration in the wind bucket?

In [ ]:
import s3fs
import xarray as xr
import ujson

endpoint_url = 'https://mghp.osn.xsede.org'
fs = s3fs.S3FileSystem(anon=True, client_kwargs={"endpoint_url": endpoint_url})

it = 1069344
face = 0
path = f"cnh-bucket-1/llc_wind/kerchunk_files/llc4320_KPPhbl-PhiBot-oceTAUX-oceTAUY-SIarea_f{face}_k0_iter_{it}.json"

with fs.open(path, "rb") as f:
    ref = ujson.load(f)

ds = xr.open_dataset(
    "reference://",
    engine="zarr",
    backend_kwargs={
        "storage_options": {
            "fo": ref,
            "remote_protocol": "s3",
            "remote_options": {
                "client_kwargs": {"endpoint_url": endpoint_url},
                "anon": True,
            },
            "asynchronous": False,
        },
        "consolidated": False,
    },
)
print(ds)
print("time value:", ds["time"].values)

# Do the no. of files match in the wind and eta buckets?

In [1]:
import s3fs
import xarray as xr
import ujson

endpoint_url = 'https://mghp.osn.xsede.org'
fs = s3fs.S3FileSystem(anon=True, client_kwargs={"endpoint_url": endpoint_url})

# --- 1. Count files in each bucket ---
wind_files = fs.glob("cnh-bucket-1/llc_wind/kerchunk_files/llc4320_*_f0_k0_iter_*.json")
surf_files = fs.glob("cnh-bucket-1/llc_surf/kerchunk_files/llc4320_*_f0_k0_iter_*.json")

print(f"Wind files (f0): {len(wind_files)}")
print(f"Surf files (f0): {len(surf_files)}")

# Extract iteration numbers from each
wind_iters = sorted(int(f.split("iter_")[1].split(".json")[0]) for f in wind_files)
surf_iters = sorted(int(f.split("iter_")[1].split(".json")[0]) for f in surf_files)

print(f"\nWind iter range: {wind_iters[0]} – {wind_iters[-1]}, step={wind_iters[1]-wind_iters[0]}")
print(f"Surf iter range: {surf_iters[0]} – {surf_iters[-1]}, step={surf_iters[1]-surf_iters[0]}")

# Check overlap
wind_set, surf_set = set(wind_iters), set(surf_iters)
print(f"\nOverlap: {len(wind_set & surf_set)} iterations")
print(f"In wind only: {len(wind_set - surf_set)}")
print(f"In surf only: {len(surf_set - wind_set)}")

Wind files (f0): 6178
Surf files (f0): 10311

Wind iter range: 179856 – 1069344, step=144
Surf iter range: 10368 – 1495008, step=144

Overlap: 6178 iterations
In wind only: 0
In surf only: 4133


In [3]:
import s3fs
import xarray as xr
import ujson
import numpy as np

endpoint_url = 'https://mghp.osn.xsede.org'
fs = s3fs.S3FileSystem(anon=True, client_kwargs={"endpoint_url": endpoint_url})

# --- Count files in each bucket ---
wind_files = fs.glob("cnh-bucket-1/llc_wind/kerchunk_files/llc4320_*_f0_k0_iter_*.json")
surf_files = fs.glob("cnh-bucket-1/llc_surf/kerchunk_files/llc4320_*_f0_k0_iter_*.json")

print(f"Wind files (f0): {len(wind_files)}")
print(f"Surf files (f0): {len(surf_files)}")

wind_iters = sorted(int(f.split("iter_")[1].split(".json")[0]) for f in wind_files)
surf_iters = sorted(int(f.split("iter_")[1].split(".json")[0]) for f in surf_files)

print(f"\nWind iter range: {wind_iters[0]} – {wind_iters[-1]}, step={wind_iters[1]-wind_iters[0]}")
print(f"Surf iter range: {surf_iters[0]} – {surf_iters[-1]}, step={surf_iters[1]-surf_iters[0]}")

wind_set, surf_set = set(wind_iters), set(surf_iters)
print(f"\nOverlap: {len(wind_set & surf_set)} iterations")
print(f"In wind only: {len(wind_set - surf_set)}")
print(f"In surf only: {len(surf_set - wind_set)}")

# --- Helper to open one kerchunk file ---
def open_kerchunk(path):
    with fs.open(path, "rb") as f:
        ref = ujson.load(f)
    return xr.open_dataset(
        "reference://", engine="zarr",
        backend_kwargs={
            "storage_options": {
                "fo": ref,
                "remote_protocol": "s3",
                "remote_options": {
                    "client_kwargs": {"endpoint_url": endpoint_url},
                    "anon": True,
                },
                "asynchronous": False,
            },
            "consolidated": False,
        },
    )

# --- Check time at first/last iteration for each bucket ---
for label, files, iters in [("Surf", surf_files, surf_iters),
                             ("Wind", wind_files, wind_iters)]:
    # Build paths for first and last iteration (face 0)
    pattern_prefix = [f for f in files if f"iter_{iters[0]}.json" in f][0]
    pattern_suffix = [f for f in files if f"iter_{iters[-1]}.json" in f][0]

    ds_first = open_kerchunk(pattern_prefix)
    ds_last  = open_kerchunk(pattern_suffix)

    t_first = ds_first["time"].values
    t_last  = ds_last["time"].values

    print(f"\n{label}:")
    print(f"  First: iter {iters[0]:>10d}  →  time = {t_first}")
    print(f"  Last:  iter {iters[-1]:>10d}  →  time = {t_last}")

# --- Check delta-t consistency ---
for label, iters in [("Surf", surf_iters), ("Wind", wind_iters)]:
    deltas = np.diff(iters)
    unique_deltas = np.unique(deltas)
    print(f"\n{label} delta-iter:")
    if len(unique_deltas) == 1:
        print(f"  Constant: {unique_deltas[0]}  ({unique_deltas[0] * 25 / 3600:.1f} hours)")
    else:
        print(f"  NOT constant! {len(unique_deltas)} unique values:")
        for d in unique_deltas:
            count = np.sum(deltas == d)
            print(f"    {d} ({d * 25 / 3600:.1f} hours) — {count} occurrences")

Wind files (f0): 6178
Surf files (f0): 10311

Wind iter range: 179856 – 1069344, step=144
Surf iter range: 10368 – 1495008, step=144

Overlap: 6178 iterations
In wind only: 0
In surf only: 4133


/home/lhoffma2/miniforge3/envs/llcngp/lib/python3.12/site-packages/zarr/storage/_fsspec.py:245: ZarrUserWarning: fs (<fsspec.implementations.reference.ReferenceFileSystem object at 0x7b8fbe86a210>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(fs=fs, path=path, read_only=read_only, allowed_exceptions=allowed_exceptions)
/home/lhoffma2/miniforge3/envs/llcngp/lib/python3.12/site-packages/zarr/storage/_fsspec.py:245: ZarrUserWarning: fs (<fsspec.implementations.reference.ReferenceFileSystem object at 0x7b8fbe04cb00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(fs=fs, path=path, read_only=read_only, allowed_exceptions=allowed_exceptions)



Surf:
  First: iter      10368  →  time = ['2011-09-13T00:00:00.000000000']
  Last:  iter    1495008  →  time = ['2012-11-15T14:00:00.000000000']


/home/lhoffma2/miniforge3/envs/llcngp/lib/python3.12/site-packages/zarr/storage/_fsspec.py:245: ZarrUserWarning: fs (<fsspec.implementations.reference.ReferenceFileSystem object at 0x7b8fbe0b1010>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(fs=fs, path=path, read_only=read_only, allowed_exceptions=allowed_exceptions)
/home/lhoffma2/miniforge3/envs/llcngp/lib/python3.12/site-packages/zarr/storage/_fsspec.py:245: ZarrUserWarning: fs (<fsspec.implementations.reference.ReferenceFileSystem object at 0x7b8fbe87f260>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(fs=fs, path=path, read_only=read_only, allowed_exceptions=allowed_exceptions)



Wind:
  First: iter     179856  →  time = ['2011-11-01T01:00:00.000000000']
  Last:  iter    1069344  →  time = ['2012-07-15T10:00:00.000000000']

Surf delta-iter:
  Constant: 144  (1.0 hours)

Wind delta-iter:
  Constant: 144  (1.0 hours)


# Compare time at iterations in the two buckets.

In [2]:
# --- 2. Compare time values at a given iteration ---
it = 1069344
face = 0

def open_kerchunk(pattern, it, face):
    path = pattern.format(face=face, it=it)
    with fs.open(path, "rb") as f:
        ref = ujson.load(f)
    return xr.open_dataset(
        "reference://", engine="zarr",
        backend_kwargs={
            "storage_options": {
                "fo": ref,
                "remote_protocol": "s3",
                "remote_options": {
                    "client_kwargs": {"endpoint_url": endpoint_url},
                    "anon": True,
                },
                "asynchronous": False,
            },
            "consolidated": False,
        },
    )

surf_pattern = "cnh-bucket-1/llc_surf/kerchunk_files/llc4320_Eta-U-V-W-Theta-Salt_f{face}_k0_iter_{it}.json"
wind_pattern = "cnh-bucket-1/llc_wind/kerchunk_files/llc4320_KPPhbl-PhiBot-oceTAUX-oceTAUY-SIarea_f{face}_k0_iter_{it}.json"

# Check if this iteration exists in both
if it in wind_set:
    ds_wind = open_kerchunk(wind_pattern, it, face)
    print(f"Wind  iter {it}: time = {ds_wind['time'].values}")
else:
    print(f"Wind  iter {it}: NOT FOUND")

if it in surf_set:
    ds_surf = open_kerchunk(surf_pattern, it, face)
    print(f"Surf  iter {it}: time = {ds_surf['time'].values}")
else:
    print(f"Surf  iter {it}: NOT FOUND")

/home/lhoffma2/miniforge3/envs/llcngp/lib/python3.12/site-packages/zarr/storage/_fsspec.py:245: ZarrUserWarning: fs (<fsspec.implementations.reference.ReferenceFileSystem object at 0x7b8fc0a6e4e0>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(fs=fs, path=path, read_only=read_only, allowed_exceptions=allowed_exceptions)


Wind  iter 1069344: time = ['2012-07-15T10:00:00.000000000']


/home/lhoffma2/miniforge3/envs/llcngp/lib/python3.12/site-packages/zarr/storage/_fsspec.py:245: ZarrUserWarning: fs (<fsspec.implementations.reference.ReferenceFileSystem object at 0x7b8fbe9ae120>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(fs=fs, path=path, read_only=read_only, allowed_exceptions=allowed_exceptions)


Surf  iter 1069344: time = ['2012-07-15T10:00:00.000000000']
